[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nekrut/bda/blob/colab/lectures/lecture9b.ipynb)

# Lecture 9b: Analyzing SRA Metadata

The [Sequence Read Archive](https://www.ncbi.nlm.nih.gov/sra) (SRA) is the largest public repository of sequencing data, mirrored by the European Nucleotide Archive (ENA). Here we analyze SARS-CoV-2 metadata to understand how sequencing platforms and library protocols were used during the pandemic.

## Setup

In [1]:
import pandas as pd

We use a pre-compiled ENA metadata snapshot hosted on [Zenodo](https://zenodo.org/records/10680776). The file contains ~800k records; we load the first 100k for speed.

In [2]:
sra = pd.read_csv(
    "https://zenodo.org/records/10680776/files/ena.tsv.gz",
    compression='gzip',
    sep="\t",
    low_memory=False,
    nrows=100000
)

## Explore the data

In [3]:
len(sra)

100000

In [4]:
sra.sample(5)

,study_accession,base_count,accession,collection_date,country,culture_collection,description,sample_collection,sample_title,sequencing_method,...,library_name,library_construction_protocol,library_layout,instrument_model,instrument_platform,isolation_source,isolate,investigation_type,collection_date_submitted,center_name
29145,PRJEB44141,245086642.0,SAMEA13601316,2022-02-14,Greece,NaN,NextSeq 500 sequencing,NaN,V7310,NaN,...,not provided,not provided,PAIRED,NextSeq 500,ILLUMINA,NaN,RNA,NaN,2022-02-14,INSTITUTE OF APPLIED BIOSCIENCES - CERTH
20701,PRJEB37886,618143432.0,SAMEA10285169,2021-09-27,United Kingdom,NaN,Illumina NovaSeq 6000 sequencing; Illumina Nov...,NaN,COG-UK/MILK-1F9F1C5,NaN,...,NT1698594A / HT-120624:E6,NaN,PAIRED,Illumina NovaSeq 6000,ILLUMINA,NaN,NaN,NaN,2021-09-27,SC
47251,PRJNA716984,6785906.0,SAMN24531803,2021-12-17,USA: Hawaii,NaN,Sequel II sequencing,NaN,CDC Sars CoV2 Sequencing Baseline Constellation,NaN,...,Unknown,Freed primers,PAIRED,Sequel II,PACBIO_SMRT,Nasal Swabs,SARS-CoV-2/Human/USA/HI-CDC-LC0436928/2021,NaN,2021-12-17,NaN
68355,PRJNA716984,3841526.0,SAMN22144621,2021-09-15,USA: Ohio,NaN,Sequel II sequencing,NaN,CDC Sars CoV2 Sequencing Baseline Constellation,NaN,...,Unknown,Freed primers,PAIRED,Sequel II,PACBIO_SMRT,Nasal Swabs,SARS-CoV-2/Human/USA/OH-CDC-LC0299062/2021,NaN,2021-09-15,NaN
44376,PRJNA716984,6963498.0,SAMN23250263,2021-11-03,USA: New York,NaN,Sequel II sequencing,NaN,CDC Sars CoV2 Sequencing Baseline Constellation,NaN,...,Unknown,Freed primers,PAIRED,Sequel II,PACBIO_SMRT,Nasal Swabs,SARS-CoV-2/Human/USA/NY-CDC-LC0364141/2021,NaN,2021-11-03,NaN


In [5]:
sra.columns.tolist()

['study_accession',
 'base_count',
 'accession',
 'collection_date',
 'country',
 'culture_collection',
 'description',
 'sample_collection',
 'sample_title',
 'sequencing_method',
 'sample_material',
 'sample_description',
 'sample_accession',
 'sample_capture_status',
 'sample_alias',
 'library_selection',
 'location',
 'run_accession',
 'read_count',
 'project_name',
 'library_source',
 'library_strategy',
 'library_name',
 'library_construction_protocol',
 'library_layout',
 'instrument_model',
 'instrument_platform',
 'isolation_source',
 'isolate',
 'investigation_type',
 'collection_date_submitted',
 'center_name']

In [6]:
sra['instrument_platform'].value_counts()

instrument_platform
ILLUMINA           81692
PACBIO_SMRT         8431
OXFORD_NANOPORE     8066
ION_TORRENT         1790
BGISEQ                18
DNBSEQ                 3
Name: count, dtype: int64

## Clean dates

In [7]:
# Convert collection_date to datetime
# errors='coerce' turns unparseable dates into NaT (Not a Time)
sra = sra.assign(collection_date=pd.to_datetime(sra['collection_date'], errors='coerce'))

In [8]:
print('Earliest entry:', sra['collection_date'].min())
print('Latest entry:', sra['collection_date'].max())

Earliest entry: 2019-12-30 00:00:00
Latest entry: 2023-01-20 00:00:00


> **⚠️ Data Quality:** Don't get surprised here — the metadata is only as good as the person who entered it. When you enter metadata for your sequencing data, pay attention!

In [9]:
# Filter to valid date range
sra = sra[
    (sra['collection_date'] >= pd.Timestamp('2020-01-01'))
    &
    (sra['collection_date'] <= pd.Timestamp('2023-12-31'))
]

## Aggregate for visualization

In [10]:
heatmap_2d = sra.groupby(
    ['instrument_platform', 'library_strategy']
).agg(
    {'run_accession': 'nunique'}
).reset_index()

heatmap_2d

,instrument_platform,library_strategy,run_accession
0,BGISEQ,AMPLICON,1
1,BGISEQ,OTHER,13
2,BGISEQ,RNA-Seq,2
3,BGISEQ,Targeted-Capture,2
4,DNBSEQ,AMPLICON,3
5,ILLUMINA,AMPLICON,78262
6,ILLUMINA,OTHER,2
7,ILLUMINA,RNA-Seq,524
8,ILLUMINA,Targeted-Capture,272
9,ILLUMINA,WCS,2


## Visualize with Altair

In [11]:
import altair as alt

In [12]:
back = alt.Chart(heatmap_2d).mark_rect(opacity=1).encode(
    x=alt.X(
        "instrument_platform:N",
        title="Instrument"
    ),
    y=alt.Y(
        "library_strategy:N",
        title="Strategy",
        axis=alt.Axis(orient='right')
    ),
    color=alt.Color(
        "run_accession:Q",
        title="# Samples",
        scale=alt.Scale(
            scheme="goldred",
            type="log"
        ),
    ),
    tooltip=[
        alt.Tooltip("instrument_platform:N", title="Machine"),
        alt.Tooltip("run_accession:Q", title="Number of runs"),
        alt.Tooltip("library_strategy:N", title="Protocol")
    ]
).properties(
    width=500,
    height=150,
    title={
        "text": ["Breakdown of datasets from ENA",
                 "by Platform and Library Strategy"],
        "subtitle": "(Sample of 100k records)"
    }
)

back

alt.Chart(...)

In [13]:
# Add text labels
front = back.mark_text(
    align="center",
    baseline="middle",
    fontSize=12,
    fontWeight="bold",
).encode(
    text=alt.Text("run_accession:Q", format=",.0f"),
    color=alt.condition(
        alt.datum.run_accession > 200,
        alt.value("white"),
        alt.value("black")
    )
)

# Combine layers
back + front

alt.LayerChart(...)